# OpenShell Sandbox — Deep Agent policy demo

A concise end-to-end walkthrough: a real LangChain Deep Agent runs on the host
with `ChatNVIDIA`, while its built-in `execute` tool runs commands inside an
OpenShell sandbox through `OpenShellSandbox`.

The flow has three explicit steps:

1. Start or connect to an OpenShell gateway.
2. Create a named OpenShell sandbox with a policy.
3. Attach the Deep Agent to that sandbox by name.

This notebook uses a named sandbox because the demo changes policy between two
runs. A short-lived Python context manager works well for per-run isolation,
but creating the sandbox with the CLI makes the image, policy, and reuse
behavior visible.

The story below is **GitHub-only policy -> agent observes mixed allowed/blocked
commands -> expand policy to PyPI -> the same agent flow observes the new access
while an unrelated API stays blocked**.


## Setup OpenShell

From a terminal in this directory, run the setup script once:

```bash
cd libs/openshell/docs/sandboxes
./setup_openshell.sh --openshell-version 0.0.72
```

The script validates the operating system, OpenShell release, Python SDK wheel compatibility, `grpcio` runtime version,
Docker/runtime readiness, Poetry environment, and Jupyter kernel registration. It creates the
`langchain-nvidia-openshell` kernel used by this notebook and writes
`openshell_setup_report.txt` beside the script.

Reload your editor window after the script finishes, then select the
**langchain-nvidia-openshell** kernel in the notebook kernel picker.


## Verify the Notebook Environment

The setup script should already have installed everything. This cell only verifies that the selected kernel is using the expected local package and OpenShell SDK versions.


## Access the NVIDIA API Catalog

This notebook runs a live `ChatNVIDIA` Deep Agent, so it needs an
`NVIDIA_API_KEY`. Create a free key from the [NVIDIA API Catalog](https://build.nvidia.com/)
and add it as `NVIDIA_API_KEY` to the repository's `.env` file. The next
cell finds and loads that file automatically, or prompts when no key is available.


In [ ]:
import os
import sys
import warnings
from pathlib import Path

from langchain_core._api.deprecation import LangChainPendingDeprecationWarning

warnings.filterwarnings(
    "ignore",
    message="The default value of `allowed_objects` will change.*",
    category=LangChainPendingDeprecationWarning,
)

# Make notebook shell commands (`!openshell ...`) use the kernel venv first.
os.environ['PATH'] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get('PATH', '')

import grpc
import openshell
import langchain_nvidia_openshell

assert tuple(map(int, grpc.__version__.split('.')[:2])) >= (1, 78), grpc.__version__
print('OpenShell SDK:', getattr(openshell, '__version__', 'unknown'))
print('grpcio:', grpc.__version__)
print('langchain-nvidia-openshell:', langchain_nvidia_openshell.__version__)


In [ ]:
import getpass
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path)
    print(f"Loaded environment from {Path(dotenv_path).resolve()}")
else:
    print("No .env file found; checking the existing environment.")

if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    print("NVIDIA_API_KEY found; ready to run the Deep Agent.")
else:
    nvapi_key = getpass.getpass("NVIDIA_API_KEY (starts with nvapi-): ")
    assert nvapi_key.startswith("nvapi-"), "NVIDIA_API_KEY must start with nvapi-"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    print("NVIDIA_API_KEY set for this notebook kernel.")


## 1. Define the Deep Agent policy audit

`create_deep_agent(..., backend=backend)` gives the agent Deep Agents' built-in
`execute` tool. Because `backend` is an `OpenShellSandbox`, every `execute` call
runs inside the policy-enforced OpenShell sandbox rather than on the host.

The helper below builds one Deep Agent against whichever sandbox session is
passed in. The sandbox lifecycle stays outside the helper: later cells create a
named sandbox with the OpenShell CLI, then attach to it with
`SandboxClient.get_session("openshell-demo")`.


In [ ]:
import os

from IPython.display import Markdown, display
from deepagents import create_deep_agent
from langchain_nvidia_ai_endpoints import ChatNVIDIA

MODEL_NAME = os.environ.get("NVIDIA_DEEP_AGENT_MODEL", "nvidia/nemotron-3-nano-30b-a3b")

AUDIT_COMMANDS = {
    "github_repo_page": (
        "curl -sSf --max-time 10 -o /dev/null -w "
        "'github.com/NVIDIA/OpenShell status=%{http_code}\\n' "
        "https://github.com/NVIDIA/OpenShell"
    ),
    "github_readme": (
        "curl -sSf --max-time 10 "
        "https://raw.githubusercontent.com/NVIDIA/OpenShell/main/README.md | wc -c"
    ),
    "pypi_openshell_version": (
        "python3 -c \"import json, urllib.request; "
        "data=json.load(urllib.request.urlopen("
        "'https://pypi.org/pypi/openshell/json', timeout=5)); "
        "print('openshell ' + data['info']['version'])\""
    ),
    "external_ip_probe": "curl -sSf --max-time 5 https://httpbin.org/ip",
}


def _shorten(text, limit=180):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "..."


def _message_content(message):
    content = getattr(message, "content", "")
    if isinstance(content, list):
        return " ".join(str(part) for part in content)
    return str(content)


def _markdown_cell(text):
    return _shorten(text).replace("|", "\\|")


def _execute_trace_markdown(result):
    commands_by_call_id = {}
    rows = ["| execute command | observed result |", "|---|---|"]
    for message in result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            if call.get("name") == "execute":
                commands_by_call_id[call.get("id")] = call.get("args", {}).get("command", "")
        if getattr(message, "type", None) == "tool" and getattr(message, "name", None) == "execute":
            command = commands_by_call_id.get(getattr(message, "tool_call_id", None), "(unknown)")
            rows.append(f"| `{_markdown_cell(command)}` | {_markdown_cell(_message_content(message))} |")
    return "\n".join(rows)


def _execute_observations(result):
    commands_by_call_id = {}
    labels_by_command = {command: label for label, command in AUDIT_COMMANDS.items()}
    observations = {}
    for message in result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            if call.get("name") == "execute":
                commands_by_call_id[call.get("id")] = call.get("args", {}).get("command", "")
        if getattr(message, "type", None) == "tool" and getattr(message, "name", None) == "execute":
            command = commands_by_call_id.get(getattr(message, "tool_call_id", None), "")
            label = labels_by_command.get(command)
            if label:
                observations[label] = _message_content(message)
    return observations


def _verified_policy_audit_markdown(result, expectations):
    observations = _execute_observations(result)
    rows = ["| check | expected | observed | verdict |", "|---|---|---|---|"]
    for label in AUDIT_COMMANDS:
        output = observations.get(label, "No matching execute result.")
        succeeded = "[Command succeeded with exit code 0]" in output
        observed = "allowed" if succeeded else "blocked"
        denial_markers = ("403", "tunnel", "failed with exit code")
        if not succeeded and any(marker in output.lower() for marker in denial_markers):
            observed += " (OpenShell policy denial)"
        verdict = "PASS" if (succeeded == (expectations[label] == "allowed")) else "CHECK"
        rows.append(
            f"| {label} | {expectations[label]} | {observed}: {_markdown_cell(output)} | {verdict} |"
        )
    return "\n".join(rows)


def _execute_result_count(result):
    return sum(
        1
        for message in result.get("messages", [])
        if getattr(message, "type", None) == "tool"
        and getattr(message, "name", None) == "execute"
    )


def _build_policy_prompt(policy_name, expectations):
    command_lines = []
    for label, command in AUDIT_COMMANDS.items():
        command_lines.append(
            f"- {label}: expected {expectations[label]}; run exactly: `{command}`"
        )
    return f"""
You are auditing OpenShell sandbox network policy: {policy_name}.

Use the `execute` tool for every command below. The `execute` tool runs inside
the OpenShell sandbox, not on the host. Do not answer from prior knowledge.
Do not combine commands; call `execute` once per listed command.

Commands:
{chr(10).join(command_lines)}

After all commands complete, briefly summarize what the commands observed.
""".strip()


def run_deep_agent_policy_audit(backend, policy_name, expectations):
    model = ChatNVIDIA(model=MODEL_NAME, temperature=0, max_completion_tokens=1600)
    agent = create_deep_agent(
        model=model,
        backend=backend,
        system_prompt=(
            "You are a careful policy-audit agent. You must use tools to inspect "
            "the live sandbox state, then summarize only what you observed."
        ),
    )
    prompt = _build_policy_prompt(policy_name, expectations)
    for attempt in range(2):
        result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
        if _execute_result_count(result) >= len(AUDIT_COMMANDS):
            break
        prompt += (
            "\n\nYour previous attempt returned before all execute results were "
            "observed. Begin with the four required execute tool calls."
        )
    else:
        raise RuntimeError("The agent did not complete all execute checks after two attempts")
    display(Markdown("### Execute Tool Trace\n" + _execute_trace_markdown(result)))
    display(Markdown("### Verified Policy Audit\n" + _verified_policy_audit_markdown(result, expectations)))
    return result


## 2. Policy A: create a GitHub-only sandbox

The demo image below keeps the notebook self-contained while satisfying
OpenShell's runtime requirements: a `sandbox` user, Python, bash, curl,
`iproute2`, and `iptables`.

The first policy allows the sandbox user to run the demo tools and permits only
`curl` traffic to `github.com` and `raw.githubusercontent.com`. GitHub reads should work; PyPI and
unrelated APIs should be denied.

The sandbox is created by name with the CLI so the policy is explicit and the
same running sandbox can be attached from Python in the next cell.


In [ ]:
%%writefile Dockerfile.openshell-demo
FROM python:3.13-slim
ENV DEBIAN_FRONTEND=noninteractive
RUN apt-get update \
    && apt-get install -y --no-install-recommends \
        bash ca-certificates curl iproute2 iptables procps \
    && rm -rf /var/lib/apt/lists/* \
    && groupadd -r sandbox \
    && useradd -r -g sandbox -d /sandbox -s /bin/bash sandbox \
    && mkdir -p /sandbox /tmp \
    && chown -R sandbox:sandbox /sandbox
WORKDIR /sandbox
USER sandbox
ENTRYPOINT ["/bin/bash"]


In [ ]:
%%writefile policy-github.yaml
version: 1

filesystem_policy:
  include_workdir: true
  read_only: [/usr, /usr/local, /lib, /etc, /app, /var/log, /proc/self, /dev/urandom]
  read_write: [/sandbox, /tmp, /dev/null]

landlock:
  compatibility: best_effort

process:
  run_as_user: sandbox
  run_as_group: sandbox

network_policies:
  github_content:
    name: github-content-readonly
    endpoints:
      - host: github.com
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
      - host: raw.githubusercontent.com
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
    binaries:
      - { path: /usr/bin/curl }


In [ ]:
import os
import subprocess
import time
from pathlib import Path

if not os.environ.get("DOCKER_HOST"):
    colima_socket = Path.home() / ".colima" / "openshell" / "docker.sock"
    if colima_socket.exists():
        os.environ["DOCKER_HOST"] = f"unix://{colima_socket}"

image = "langchain-nvidia-openshell-demo:0.0.72"
if subprocess.run(["docker", "image", "inspect", image], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
    subprocess.run(["docker", "build", "-t", image, "-f", "Dockerfile.openshell-demo", "."], check=True)

subprocess.run(["openshell", "sandbox", "delete", "openshell-demo"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

log_path = Path("/tmp/openshell-demo-github.log")
log = log_path.open("wb")
subprocess.Popen(
    [
        "openshell", "sandbox", "create",
        "--name", "openshell-demo",
        "--from", image,
        "--policy", "policy-github.yaml",
        "--no-tty", "--", "sleep", "infinity",
    ],
    stdin=subprocess.DEVNULL,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    close_fds=True,
)
log.close()

for _ in range(60):
    status = subprocess.run(["openshell", "sandbox", "list"], capture_output=True, text=True).stdout
    if "openshell-demo" in status and "Ready" in status:
        print("openshell-demo Ready")
        break
    time.sleep(1)
else:
    raise TimeoutError(log_path.read_text(errors="replace"))


## 3. Run the Deep Agent — GitHub works, others are blocked

The Deep Agent receives the named OpenShell sandbox as its backend and calls its
built-in `execute` tool. Under the GitHub-only policy, it should observe that
GitHub commands work while PyPI and `httpbin.org` are blocked.

`OpenShellSandbox` does not create or delete this sandbox. It only adapts the
existing `openshell-demo` session to the LangChain Deep Agents sandbox
interface.


In [ ]:
import openshell
from langchain_nvidia_openshell import OpenShellSandbox

client = openshell.SandboxClient.from_active_cluster()
backend = OpenShellSandbox(sandbox=client.get_session("openshell-demo"))

policy_a_result = run_deep_agent_policy_audit(
    backend,
    "Policy A: GitHub-only",
    {
        "github_repo_page": "allowed",
        "github_readme": "allowed",
        "pypi_openshell_version": "blocked",
        "external_ip_probe": "blocked",
    },
)
client.close()


## 4. Expand the policy, recreate the sandbox

The second policy keeps GitHub access, adds PyPI access for Python, and still
does not allow `httpbin.org`.

OpenShell can hot-reload network policy with `openshell policy set`, but this
notebook recreates the sandbox so every policy layer is applied from a clean
starting point and the before/after comparison stays easy to inspect.


In [ ]:
!openshell sandbox delete openshell-demo


In [ ]:
%%writefile policy-expanded.yaml
version: 1

filesystem_policy:
  include_workdir: true
  read_only: [/usr, /usr/local, /lib, /etc, /app, /var/log, /proc/self, /dev/urandom]
  read_write: [/sandbox, /tmp, /dev/null]

landlock:
  compatibility: best_effort

process:
  run_as_user: sandbox
  run_as_group: sandbox

network_policies:
  github_content:
    name: github-content-readonly
    endpoints:
      - host: github.com
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
      - host: raw.githubusercontent.com
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
    binaries:
      - { path: /usr/bin/curl }
  pypi_api:
    name: pypi-readonly
    endpoints:
      - host: pypi.org
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
    binaries:
      - { path: /usr/local/bin/python3 }
      - { path: /usr/local/bin/python3.13 }


In [ ]:
import os
import subprocess
import time
from pathlib import Path

if not os.environ.get("DOCKER_HOST"):
    colima_socket = Path.home() / ".colima" / "openshell" / "docker.sock"
    if colima_socket.exists():
        os.environ["DOCKER_HOST"] = f"unix://{colima_socket}"

image = "langchain-nvidia-openshell-demo:0.0.72"
if subprocess.run(["docker", "image", "inspect", image], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
    subprocess.run(["docker", "build", "-t", image, "-f", "Dockerfile.openshell-demo", "."], check=True)

log_path = Path("/tmp/openshell-demo-expanded.log")
log = log_path.open("wb")
subprocess.Popen(
    [
        "openshell", "sandbox", "create",
        "--name", "openshell-demo",
        "--from", image,
        "--policy", "policy-expanded.yaml",
        "--no-tty", "--", "sleep", "infinity",
    ],
    stdin=subprocess.DEVNULL,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    close_fds=True,
)
log.close()

for _ in range(60):
    status = subprocess.run(["openshell", "sandbox", "list"], capture_output=True, text=True).stdout
    if "openshell-demo" in status and "Ready" in status:
        print("openshell-demo Ready")
        break
    time.sleep(1)
else:
    raise TimeoutError(log_path.read_text(errors="replace"))


## 5. Re-run the Deep Agent — PyPI is added, httpbin stays blocked

Same Deep Agent pattern, same adapter, new sandbox policy. The agent attaches to
the recreated `openshell-demo` sandbox and should now observe that PyPI succeeds
while `httpbin.org` remains blocked.


In [ ]:
client = openshell.SandboxClient.from_active_cluster()
backend = OpenShellSandbox(sandbox=client.get_session("openshell-demo"))

policy_b_result = run_deep_agent_policy_audit(
    backend,
    "Policy B: GitHub + PyPI",
    {
        "github_repo_page": "allowed",
        "github_readme": "allowed",
        "pypi_openshell_version": "allowed",
        "external_ip_probe": "blocked",
    },
)
client.close()


## 6. Cleanup

Delete the sandbox, remove the policy files, and confirm a clean slate.


In [ ]:
%%bash
openshell sandbox delete openshell-demo || true
rm -f policy-github.yaml policy-expanded.yaml Dockerfile.openshell-demo
echo "--- sandboxes ---"
openshell sandbox list
echo "--- policy files left behind ---"
ls policy-github.yaml policy-expanded.yaml Dockerfile.openshell-demo 2>/dev/null || echo "(none)"


## Sources

- [OpenShell — sandbox-policy-quickstart](https://github.com/NVIDIA/OpenShell/tree/main/examples/sandbox-policy-quickstart)
  — the deny-then-allow demo this notebook mirrors.
- [OpenShell policy schema reference](https://docs.nvidia.com/openshell/latest/reference/policy-schema)
  — every YAML field in the policy files above.
- [LangChain Deep Agents — sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes)
  — the sandbox-as-tool pattern.
- [`langchain-nvidia-openshell` README](../../README.md) — the full API
  surface, setup flow, and policy lifecycle notes.
